# 08 - Fusion

Feature-level fusion: the 64-d penultimate layer is concatenated with the tabular matrix and XGBoost is trained on the concatenation. Set `embedding_source.run_id` in `fusion.yaml` to the CNN-BiLSTM run that produced the weights.

In [ ]:
# --- standard header: every notebook starts with exactly this ---
from google.colab import drive; drive.mount('/content/drive')

REPO = '/content/secure-dns-trust-ai'
!git -C {REPO} pull -q 2>/dev/null || git clone -q https://github.com/sandesh20lamichhane/secure-dns-trust-ai.git {REPO}

import sys, os; sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'
%load_ext autoreload
%autoreload 2

from src.utils import config, manifest, seeds, io
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
import pandas as pd, torch
from src.models.cnn_bilstm import CharEncoder, CNNBiLSTM
from src.models import fusion, xgb as xgbm
from src.evaluate import splits, metrics, predictions
from src.features.build import to_matrix

cfg = config.load('fusion'); nn_cfg = config.load('cnn_bilstm')
src_run = cfg['embedding_source']['run_id']
assert src_run, 'set embedding_source.run_id in configs/fusion.yaml'

In [ ]:
enc = CharEncoder(nn_cfg['input']['charset'], nn_cfg['input']['max_length'])
net = CNNBiLSTM(enc.vocab_size, **nn_cfg['model'])
net.load_state_dict(torch.load(f"{P['artifacts']['models']}/{src_run}.pt"))

df = pd.read_parquet(f"{P['data']['features']}/fused_v1.parquet")
emb = fusion.embed_domains(net, enc, df['domain'].values)
df = fusion.fuse(df, emb, df['domain'].values)

In [ ]:
split = splits.load_split(P['data']['splits'], cfg['split']['name'])
tr, va, te = splits.apply_split(df, split)
Xtr, ytr, _ = to_matrix(tr); Xva, yva, _ = to_matrix(va); Xte, yte, _ = to_matrix(te)
model = xgbm.fit(xgbm.build(cfg['model'], ytr), Xtr, ytr, Xva, yva)
scores = model.predict_proba(Xte)[:, 1]
m = metrics.evaluate(yte, scores); m

In [ ]:
counter = manifest.next_counter(P['manifest'])
run_id = manifest.make_run_id('fusion', cfg['split']['name'], cfg['seed'], counter)
model.save_model(f"{P['artifacts']['models']}/{run_id}.json")
predictions.save(run_id, P['artifacts']['predictions'], te['domain'], yte, scores)
manifest.record(P['manifest'], run_id, 'fusion', cfg, cfg['split']['name'],
                split['split_file'], m, cfg['seed'], repo_root=REPO,
                notes=f'embedding from {src_run}')
print(run_id)